# Uji CLI dengan NER dan pytest

Clone repo, download `ner_checkpoints_seed42.zip` (checkpoint NER entitas dan CLAIM) dari Drive, menjalankan CLI, lalu `pytest`. yang diuji di sini: NER, aturan kueri retrieval, dan alur CLI

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/danenftyessir/indonesian-hoax-checker.git"
BRANCH = "feat/hoax-classification"
# ner_checkpoints_seed42.zip di hasil_training/ner_20261112-1801
NER_ZIP_FILE_ID = "1c_ouOXwCIdMgqD2_rJzF_oXL-5qS0oEN"

from google.colab import auth
auth.authenticate_user()

## 2. Clone dan install dependensi

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    url = REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
%cd /content/repo
!git pull -q
!git log --oneline -3
!pip install -q -e ".[training]" pytest

## 3. Download dan ekstrak checkpoint NER

In [ ]:
import io, zipfile
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

drive = build("drive", "v3")
with io.FileIO("/content/ner_checkpoints.zip", "wb") as f:
    dl = MediaIoBaseDownload(f, drive.files().get_media(fileId=NER_ZIP_FILE_ID, supportsAllDrives=True))
    done = False
    while not done:
        _, done = dl.next_chunk()
z = zipfile.ZipFile("/content/ner_checkpoints.zip")
names = z.namelist()
print(len(names), "berkas; contoh:", names[:4])
z.extractall("." if any(n.startswith("artifacts/") for n in names) else "artifacts/checkpoints/ner")
!ls artifacts/checkpoints/ner
!cat configs/pipeline.yaml

## 4. CLI: NER
Baris `NER` tidak lagi bertanda `[STUB]` bila checkpoint ditemukan. Retrieval, stance, dan klasifikasi tetap stub.

In [ ]:
!python -m indonesian_hoax_checker "Presiden Prabowo menandatangani Surat Keputusan baru di Jakarta." -v
!python -m indonesian_hoax_checker "Beredar pesan berantai yang mengeklaim Menteri Keuangan Purbaya akan menaikkan harga BBM subsidi pada 1 Desember 2026." -v
!python -m indonesian_hoax_checker "Pemerintah menghentikan bantuan" --task ner

## 5. Aturan kueri retrieval (core_claim + entitas PER/ORG/LOC)

In [ ]:
import sys
sys.path.insert(0, "/content/repo/src")   
from indonesian_hoax_checker.pipeline.orchestrator import run_pipeline, retrieval_query, input_claim
text = "Beredar pesan berantai yang mengeklaim Menteri Keuangan Purbaya akan menaikkan harga BBM subsidi pada 1 Desember 2026."
res = run_pipeline(text, "all")
print("stub stages:", res.stubs)
print("entitas   :", [(e["text"], e["label"]) for e in res.ner["entities"]])
print("core_claim:", repr(res.ner["core_claim"]))
print("kueri     :", repr(retrieval_query(text, res.ner)))
print("klaim stance:", repr(input_claim(text, res.ner)))

## 6. pytest

In [ ]:
!python -m pytest -q --no-header -p no:cacheprovider 2>&1 | tail -25

## 7. Classifier IndoBERT di CLI
Copy checkpoint demo dari Drive ke lokasi yang dibaca `configs/pipeline.yaml`, lalu menjalankan CLI

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
DEMO_MODEL_DIR = "/content/drive/MyDrive/SEMESTER 7/NLP/Tubes/indobert_demo/model/seed13"  
DEST = "artifacts/checkpoints/classifier/indobert_seed13"
os.makedirs(os.path.dirname(DEST), exist_ok=True)
if not os.path.exists(DEST):
    shutil.copytree(DEMO_MODEL_DIR, DEST)
!ls {DEST}
!git status --short configs/pipeline.yaml
!grep -n -A4 "^classification" configs/pipeline.yaml

In [ ]:
!python -m indonesian_hoax_checker "Presiden Prabowo menandatangani Surat Keputusan baru di Jakarta." -v
!python -m indonesian_hoax_checker "Beredar pesan berantai yang mengeklaim Menteri Keuangan Purbaya akan menaikkan harga BBM subsidi pada 1 Desember 2026." -v
!python -m indonesian_hoax_checker "Pemerintah menghentikan bantuan" --task classify
!python -m indonesian_hoax_checker "Tautan Pendaftaran BLT UMKM 2026 Sebesar Rp50 Juta" --task classify